# Tests de mini-funcionalidades de IO flows

Este notebook se usa para probar helpers y bloques internos de `write_flows()` y `read_flows()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados

## Bloque 0. Preparación

### 0.1 Imports generales

Qué prepara: imports básicos, detección de `pyarrow` y utilidades de filesystem para los tests con Parquet.

In [1]:
import copy
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import types as patypes
HAS_PYARROW = True

### 0.2 Imports del módulo

Qué prepara: imports de clases y helpers reales de `pylondrina.io.flows`.

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.reports import Issue
from pylondrina.errors import ExportError

from pylondrina.io.flows import (
    WriteFlowsOptions,
    ReadFlowsOptions,
    _validate_write_contract,
    _freeze_flow_write_snapshot,
    _create_flows_staging_dir,
    _write_flows_table_to_staging,
    _write_optional_flow_to_trips_to_staging,
    _write_flow_sidecar_to_staging,
    _assert_flows_staging_complete,
    _commit_staged_flow_bundle,
    _validate_read_layout,
    _load_flow_sidecar,
    _recover_flow_read_state,
    _read_flows_table,
    _read_optional_flow_to_trips,
    _build_read_summary,
    _normalize_flows_artifact_root_for_write,
    _resolve_flows_artifact_root_for_read,
    _resolve_flows_artifact_paths,
    _build_write_flows_summary,
    _assert_json_safe,
    _cleanup_staging_dir,
    _append_event,
    _prepare_dataframe_for_parquet,
    _ensure_dataset_id,
    _new_dataset_id,
    _new_artifact_id,
    _extract_validated_flag,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para assertions y mensajes, análogas al notebook de trips.

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_dumpable(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, f"errors esperado={errors}, actual={counts['error']}"
    if warnings is not None:
        assert counts["warning"] == warnings, f"warnings esperado={warnings}, actual={counts['warning']}"
    if info is not None:
        assert counts["info"] == info, f"info esperado={info}, actual={counts['info']}"

### 0.4 Configuración visual

Qué prepara: display más cómodo.

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para `FlowDataset`, sidecar y artefactos persistidos.

In [5]:
HELPER_ROOT = Path("./tmp_helper_flows")

def reset_helper_root() -> Path:
    if HELPER_ROOT.exists():
        shutil.rmtree(HELPER_ROOT)
    HELPER_ROOT.mkdir(parents=True, exist_ok=True)
    return HELPER_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = HELPER_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def make_flows_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f2", "f3"],
            "origin_h3_index": ["881111111111111", "882222222222222", "883333333333333"],
            "destination_h3_index": ["884444444444444", "885555555555555", "886666666666666"],
            "flow_count": [2, 1, 3],
            "flow_value": [2.0, 1.0, 4.5],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": [
                "2026-01-01T08:00:00Z",
                "2026-01-01T09:00:00Z",
                "2026-01-01T10:00:00Z",
            ],
            "window_end_utc": [
                "2026-01-01T08:59:59Z",
                "2026-01-01T09:59:59Z",
                "2026-01-01T10:59:59Z",
            ],
        }
    )


def make_flow_to_trips_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f1", "f2"],
            "movement_id": ["m1", "m2", "m3"],
        }
    )


def make_flowdataset(
    *,
    validated: bool = False,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
    include_aux: bool = True,
    aggregation_spec: dict | None = None,
    metadata_extra: dict | None = None,
    provenance: dict | None = None,
) -> FlowDataset:
    metadata = {
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-06T00:00:00Z",
                "parameters": {"h3_resolution": 8},
                "summary": {"n_flows": 3},
                "issues_summary": {"error": 0, "warning": 0, "info": 0},
            }
        ],
        "is_validated": validated,
    }
    if include_dataset_id:
        metadata["dataset_id"] = "dset_existing"
    if include_artifact_id:
        metadata["artifact_id"] = "art_existing"
    if metadata_extra:
        metadata.update(copy.deepcopy(metadata_extra))

    if aggregation_spec is None:
        aggregation_spec = {
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
        }

    if provenance is None:
        provenance = {
            "derived_from": [
                {"type": "trips", "dataset_id": "trip_dset_001"},
            ],
            "prior_events_summary": {"build_flows": 1},
        }

    return FlowDataset(
        flows=make_flows_df(),
        flow_to_trips=make_flow_to_trips_df() if include_aux else None,
        aggregation_spec=copy.deepcopy(aggregation_spec),
        source_trips="SENTINEL_SOURCE_TRIPS_ONLY_IN_MEMORY",
        metadata=copy.deepcopy(metadata),
        provenance=copy.deepcopy(provenance),
    )


def make_sidecar_payload(*, include_flow_to_trips: bool = True) -> dict:
    payload = {
        "dataset_type": "flows",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": "parquet",
            "options": {"compression": "snappy"},
        },
        "dataset_id": "dset_sidecar",
        "artifact_id": "art_sidecar",
        "files": {
            "data": "flows.parquet",
            "metadata": "flows.metadata.json",
            "flow_to_trips": "flow_to_trips.parquet" if include_flow_to_trips else None,
        },
        "aggregation_spec": {
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
        },
        "provenance": {
            "derived_from": [{"type": "trips", "dataset_id": "trip_dset_001"}],
        },
        "metadata": {
            "dataset_id": "dset_sidecar",
            "artifact_id": "art_sidecar",
            "is_validated": True,
            "events": [],
        },
        "tables": {
            "flows": {
                "n_rows": 3,
                "n_cols": len(make_flows_df().columns),
                "columns": list(make_flows_df().columns),
            },
            "flow_to_trips": {
                "n_rows": 3,
                "n_cols": len(make_flow_to_trips_df().columns),
                "columns": list(make_flow_to_trips_df().columns),
            } if include_flow_to_trips else None,
        },
    }
    return payload


def materialize_minimal_formal_flow_artifact(root: Path, *, with_aux: bool = True):
    root.mkdir(parents=True, exist_ok=True)
    paths = _resolve_flows_artifact_paths(root)

    make_flows_df().to_parquet(paths.data_path, index=False)
    if with_aux:
        make_flow_to_trips_df().to_parquet(paths.flow_to_trips_path, index=False)

    payload = make_sidecar_payload(include_flow_to_trips=with_aux)
    paths.sidecar_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return paths, payload


root = reset_helper_root()
print("HELPER_ROOT =", root.resolve())
show_ok("Bloque 1 - fixtures mínimas")

HELPER_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\io_flows\tmp_helper_flows
OK - Bloque 1 - fixtures mínimas


## Bloque 2. Helpers internos de uso general

### Test 2.1 - `_resolve_flows_artifact_paths`

Qué prueba: que el layout formal se resuelva siempre a `flows.parquet`, `flows.metadata.json` y `flow_to_trips.parquet`.


In [6]:
root = Path("tmp_demo_flows_artifact")
paths = _resolve_flows_artifact_paths(root)

assert paths.root_dir == root
assert paths.data_path == root / "flows.parquet"
assert paths.sidecar_path == root / "flows.metadata.json"
assert paths.flow_to_trips_path == root / "flow_to_trips.parquet"

show_ok("Test 2.1 - _resolve_flows_artifact_paths")

OK - Test 2.1 - _resolve_flows_artifact_paths


### Test 2.2 - normalización/fallback `.golondrina`

Qué prueba:
- write normaliza el sufijo cuando corresponde,
- read hace fallback a `.golondrina` solo si el path exacto no existe.

In [7]:
# write: agrega sufijo si corresponde
p = _normalize_flows_artifact_root_for_write("demo_artifact", normalize_artifact_dir=True)
assert p.name.endswith(".golondrina")

# write: no agrega sufijo si normalize_artifact_dir=False
p2 = _normalize_flows_artifact_root_for_write("demo_artifact", normalize_artifact_dir=False)
assert p2.name == "demo_artifact"

# read: fallback al artefacto con sufijo
case_dir = make_case_dir("case_02_suffix_fallback")
artifact_exact = case_dir / "artifact_demo"
artifact_real = case_dir / "artifact_demo.golondrina"
artifact_real.mkdir(parents=True, exist_ok=True)

resolved = _resolve_flows_artifact_root_for_read(artifact_exact)
assert resolved == artifact_real

show_ok("Test 2.2 - normalización/fallback .golondrina")

OK - Test 2.2 - normalización/fallback .golondrina


### Test 2.3 - `_assert_json_safe`

Qué prueba: acepta payload JSON-safe y falla cuando el bloque no es serializable.

In [8]:
issues_ok = []
_assert_json_safe(
    {"a": 1, "b": ["x", "y"]},
    label="payload_ok",
    issues=issues_ok,
    path=Path("/tmp/fake"),
    artifact="flows.metadata.json",
)
assert issues_ok == []

issues_bad = []
try:
    _assert_json_safe(
        {"bad": {1, 2, 3}},   # set no es JSON-safe
        label="payload_bad",
        issues=issues_bad,
        path=Path("/tmp/fake"),
        artifact="flows.metadata.json",
    )
    raise AssertionError("Debió fallar por payload no serializable")
except ExportError:
    assert_issue_present(issues_bad, "WRITE_FLOWS.SNAPSHOT.NOT_JSON_SERIALIZABLE")

show_ok("Test 2.3 - _assert_json_safe")

OK - Test 2.3 - _assert_json_safe


### Test 2.4 - `_ensure_dataset_id`, IDs nuevos y `_extract_validated_flag`

Qué prueba:
- preservación,
- creación,
- regeneración,
- lectura del flag top-level y fallback legacy interno.

In [9]:
dataset_id, status = _ensure_dataset_id({"dataset_id": "dset_ok"})
assert dataset_id == "dset_ok"
assert status == "preserved"

dataset_id2, status2 = _ensure_dataset_id({})
assert isinstance(dataset_id2, str) and dataset_id2.startswith("dset_")
assert status2 == "created"

dataset_id3, status3 = _ensure_dataset_id({"dataset_id": ""})
assert isinstance(dataset_id3, str) and dataset_id3.startswith("dset_")
assert status3 == "regenerated"

assert _extract_validated_flag({"is_validated": True}) is True
assert _extract_validated_flag({"is_validated": False}) is False
assert _extract_validated_flag({"flags": {"validated": True}}) is True
assert _extract_validated_flag({}) is False

art_id = _new_artifact_id()
assert art_id.startswith("art_")

show_ok("Test 2.4 - dataset_id / artifact_id / validated_flag")

OK - Test 2.4 - dataset_id / artifact_id / validated_flag



### Test 2.5 - `_append_event`

Qué prueba: append-only sin mutar el input original.

In [10]:
metadata = {"events": [{"op": "build_flows"}], "x": 1}
event = {"op": "write_flows", "summary": {"n_flows": 3}}

metadata_out = _append_event(metadata, event)

assert metadata is not metadata_out
assert len(metadata["events"]) == 1
assert len(metadata_out["events"]) == 2
assert metadata_out["events"][-1]["op"] == "write_flows"
assert metadata_out["x"] == 1

show_ok("Test 2.5 - _append_event")

OK - Test 2.5 - _append_event


## Bloque 3. Mini-funcionalidad de Parquet y diagnóstico de categóricos

Verifica que write_flows convierta a category los campos de segmentación
    (`group_by`) antes de escribir Parquet, y que PyArrow los materialice con
    type dictionary cuando el engine está disponible.

In [11]:
import tempfile
import pandas as pd
from pylondrina.io.flows import (
    _collect_flow_parquet_categorical_fields,
    _prepare_dataframe_for_parquet,
    _write_flows_table_to_staging,
)
import pyarrow.parquet as pq

df = pd.DataFrame(
    {
        "flow_id": ["f1", "f2", "f3", "f4"],
        "origin_h3_index": ["81", "81", "82", "82"],
        "destination_h3_index": ["91", "92", "91", "92"],
        "flow_count": [10, 20, 30, 40],
        "flow_value": [10.0, 20.0, 30.0, 40.0],
        "mode": ["bus", "bus", "metro", "metro"],
    }
)
aggregation_spec = {"group_by": ["mode"]}

# 1) Primero se comprueba la mini-función que decide qué columnas forzar.
categorical_fields = _collect_flow_parquet_categorical_fields(
    df,
    aggregation_spec=aggregation_spec,
)
assert categorical_fields == ["mode"]

# 2) Luego se comprueba la preparación previa al write.
prepared = _prepare_dataframe_for_parquet(df, categorical_fields=categorical_fields)
assert str(prepared["mode"].dtype) == "category"
assert list(prepared["mode"].cat.categories) == ["bus", "metro"]

# 3) Finalmente se verifica el efecto observable en el parquet escrito.
issues = []
with tempfile.TemporaryDirectory() as tmpdir:
    parquet_path = Path(tmpdir) / "flows.parquet"
    _write_flows_table_to_staging(
        df,
        parquet_path,
        storage_format="parquet",
        parquet_compression="snappy",
        aggregation_spec=aggregation_spec,
        issues=issues,
        destination_path=Path(tmpdir),
    )

    assert parquet_path.exists()
    assert issues == []

    table = pq.read_table(parquet_path)
    mode_type = table.schema.field("mode").type

    # PyArrow debería observar dictionary-encoding por venir desde pandas.Categorical.
    assert "dictionary" in str(mode_type).lower(), mode_type

## Bloque 4. Helpers principales de Write flows

### Test 4.1 - `_validate_write_contract` happy path

Qué prueba: caso correcto sin excepción.

In [12]:
flows = make_flowdataset()
issues = []

_validate_write_contract(
    flows,
    Path("/tmp/fake_artifact"),
    WriteFlowsOptions(),
    issues=issues,
)

assert issues == []

show_ok("Test 4.1 - _validate_write_contract happy path")

OK - Test 4.1 - _validate_write_contract happy path


### Test 4.2 - `_validate_write_contract` fatal por dataset inválido

Qué prueba: precondición básica más importante.

In [13]:
issues = []
try:
    _validate_write_contract(
        object(),   # no FlowDataset
        Path("/tmp/fake_artifact"),
        WriteFlowsOptions(),
        issues=issues,
    )
    raise AssertionError("Debió fallar por dataset inválido")
except ExportError:
    assert_issue_present(issues, "WRITE_FLOWS.INPUT.INVALID_DATASET")

show_ok("Test 4.2 - _validate_write_contract fatal invalid dataset")

OK - Test 4.2 - _validate_write_contract fatal invalid dataset


### Test 4.3 - `_validate_write_contract` fatal por storage_format no soportado

Qué prueba: cierre de backend en v1.1.

In [14]:
flows = make_flowdataset()
issues = []

try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_artifact"),
        WriteFlowsOptions(storage_format="parquet"),  # baseline
        issues=issues,
    )
except Exception as exc:
    raise AssertionError(f"No debía fallar baseline: {exc}") from exc

issues_bad = []
try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_artifact"),
        WriteFlowsOptions(storage_format="feather"),  # type: ignore[arg-type]
        issues=issues_bad,
    )
    raise AssertionError("Debió fallar por storage_format no soportado")
except ExportError:
    assert_issue_present(issues_bad, "WRITE_FLOWS.OPTIONS.UNSUPPORTED_STORAGE_FORMAT")

show_ok("Test 4.3 - _validate_write_contract storage_format")

OK - Test 4.3 - _validate_write_contract storage_format


### Test 4.4 - `_freeze_flow_write_snapshot`

Qué prueba:
- creación/regeneración de `dataset_id`,
- `artifact_id` nuevo,
- sidecar oficial top-level,
- warning cuando se pidió `flow_to_trips` pero no existe,
- y que `source_trips` no entra al sidecar.

In [15]:
flows = make_flowdataset(
    include_dataset_id=False,
    include_artifact_id=False,
    include_aux=False,
    validated=False,
)

paths = _resolve_flows_artifact_paths(Path("/tmp/fake_write.golondrina"))
snapshot = _freeze_flow_write_snapshot(
    flows,
    paths,
    WriteFlowsOptions(write_flow_to_trips=True),
    existing_issues=[],
)

assert snapshot.dataset_id_status == "created"
assert snapshot.dataset_id.startswith("dset_")
assert snapshot.artifact_id.startswith("art_")

assert snapshot.files_written == ["flows.parquet", "flows.metadata.json"]
assert snapshot.n_flow_to_trips is None

assert snapshot.sidecar_payload["dataset_type"] == "flows"
assert snapshot.sidecar_payload["format"] == "golondrina"
assert snapshot.sidecar_payload["layout_version"] == "1.1"
assert snapshot.sidecar_payload["storage"]["format"] == "parquet"
assert snapshot.sidecar_payload["files"]["data"] == "flows.parquet"
assert snapshot.sidecar_payload["files"]["metadata"] == "flows.metadata.json"
assert snapshot.sidecar_payload["files"]["flow_to_trips"] is None

assert snapshot.metadata_for_persist["dataset_id"] == snapshot.dataset_id
assert snapshot.metadata_for_persist["artifact_id"] == snapshot.artifact_id
assert snapshot.metadata_for_persist["is_validated"] is False
assert snapshot.metadata_for_persist["events"][-1]["op"] == "write_flows"

assert "source_trips" not in json.dumps(snapshot.sidecar_payload, ensure_ascii=False)

assert_issue_present(snapshot.issues, "WRITE_FLOWS.METADATA.DATASET_ID_CREATED")
assert_issue_present(snapshot.issues, "WRITE_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING")

assert_json_dumpable(snapshot.sidecar_payload, "sidecar_payload")
show_ok("Test 4.4 - _freeze_flow_write_snapshot")

OK - Test 4.4 - _freeze_flow_write_snapshot


### Test 4.5 - `_create_flows_staging_dir` y `_cleanup_staging_dir`

Qué prueba: creación de staging hermano y cleanup normal.

In [16]:
with tempfile.TemporaryDirectory() as td:
    final_dir = Path(td) / "artifact_final.golondrina"
    issues = []

    staging_dir = _create_flows_staging_dir(final_dir, issues=issues)

    assert staging_dir.exists()
    assert staging_dir.is_dir()
    assert staging_dir.parent == final_dir.parent
    assert issues == []

    _cleanup_staging_dir(
        staging_dir,
        final_dir,
        ["flows.parquet", "flows.metadata.json"],
        issues,
    )
    assert not staging_dir.exists()

show_ok("Test 4.5 - _create_flows_staging_dir + _cleanup_staging_dir")

OK - Test 4.5 - _create_flows_staging_dir + _cleanup_staging_dir


### Test 4.6 - `_write_flow_sidecar_to_staging` + `_load_flow_sidecar`

Qué prueba: round-trip de sidecar válido.

In [17]:
case_dir = make_case_dir("case_04_sidecar_roundtrip")
staging = case_dir / "staging"
staging.mkdir(parents=True, exist_ok=True)

sidecar_path = staging / "flows.metadata.json"
payload = make_sidecar_payload(include_flow_to_trips=True)

issues_write = []
_write_flow_sidecar_to_staging(
    payload,
    sidecar_path,
    issues=issues_write,
    destination_path=case_dir,
)
assert sidecar_path.exists()
assert issues_write == []

issues_load = []
loaded = _load_flow_sidecar(
    sidecar_path,
    strict=False,
    issues=issues_load,
    destination_path=case_dir,
)

assert loaded["dataset_type"] == "flows"
assert loaded["layout_version"] == "1.1"
assert loaded["storage"]["format"] == "parquet"
assert loaded["files"]["data"] == "flows.parquet"
assert loaded["metadata"]["dataset_id"] == "dset_sidecar"
assert issues_load == []

show_ok("Test 4.6 - sidecar write/load round-trip")

OK - Test 4.6 - sidecar write/load round-trip


### Test 4.7 - `_assert_flows_staging_complete`

Qué prueba: acepta staging completo y falla si falta un artefacto esperado.

In [18]:
# Caso completo
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_ok"
    staging_root.mkdir()
    paths = _resolve_flows_artifact_paths(staging_root)
    paths.data_path.touch()
    paths.sidecar_path.touch()

    issues = []
    _assert_flows_staging_complete(
        paths,
        expected_files=["flows.parquet", "flows.metadata.json"],
        issues=issues,
        destination_path=staging_root,
    )
    assert issues == []

# Caso incompleto
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_bad"
    staging_root.mkdir()
    paths = _resolve_flows_artifact_paths(staging_root)
    paths.data_path.touch()

    issues = []
    try:
        _assert_flows_staging_complete(
            paths,
            expected_files=["flows.parquet", "flows.metadata.json"],
            issues=issues,
            destination_path=staging_root,
        )
        raise AssertionError("Debió fallar por staging incompleto")
    except ExportError:
        assert_issue_present(issues, "WRITE_FLOWS.IO.STAGING_INCOMPLETE")

show_ok("Test 4.7 - _assert_flows_staging_complete")

OK - Test 4.7 - _assert_flows_staging_complete


### Test 4.8 - `_commit_staged_flow_bundle` success

Qué prueba: commit correcto desde staging al directorio final.

In [19]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact_final.golondrina"
    staging = parent / "staging"

    staging.mkdir()
    (staging / "flows.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "flows.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    _commit_staged_flow_bundle(
        staging,
        final_dir,
        mode="error_if_exists",
        files_written=["flows.parquet", "flows.metadata.json"],
        issues=issues,
    )

    assert final_dir.exists()
    assert (final_dir / "flows.parquet").exists()
    assert (final_dir / "flows.metadata.json").exists()
    assert not staging.exists()
    assert issues == []

show_ok("Test 4.8 - _commit_staged_flow_bundle success")

OK - Test 4.8 - _commit_staged_flow_bundle success


### Test 4.9 - `_commit_staged_flow_bundle` fatal por colisión

Qué prueba: `mode="error_if_exists"`.

In [20]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact_final.golondrina"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "flows.parquet").write_text("dummy parquet placeholder", encoding="utf-8")
    (staging / "flows.metadata.json").write_text("{}", encoding="utf-8")

    issues = []
    try:
        _commit_staged_flow_bundle(
            staging,
            final_dir,
            mode="error_if_exists",
            files_written=["flows.parquet", "flows.metadata.json"],
            issues=issues,
        )
        raise AssertionError("Debió fallar por destino ya existente")
    except ExportError:
        assert_issue_present(issues, "WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS")

show_ok("Test 4.9 - _commit_staged_flow_bundle collision")

OK - Test 4.9 - _commit_staged_flow_bundle collision


### Test 4.10 - `_build_write_flows_summary`

Qué prueba: summary mínimo estable.

In [21]:
summary = _build_write_flows_summary(
    n_flows=3,
    n_flow_to_trips=7,
    path=Path("/tmp/artifact.golondrina"),
    dataset_id="dset_001",
    artifact_id="art_001",
    files_written=["flows.parquet", "flows.metadata.json", "flow_to_trips.parquet"],
)

assert summary["n_flows"] == 3
assert summary["n_flow_to_trips"] == 7
assert Path(summary["path"]) == Path("/tmp/artifact.golondrina")
assert summary["dataset_id"] == "dset_001"
assert summary["artifact_id"] == "art_001"
assert summary["files_written"] == [
    "flows.parquet",
    "flows.metadata.json",
    "flow_to_trips.parquet",
]
assert_json_dumpable(summary, "write_flows_summary")

show_ok("Test 4.10 - _build_write_flows_summary")

OK - Test 4.10 - _build_write_flows_summary


### Test 4.11 - `_write_flows_table_to_staging` y `_write_optional_flow_to_trips_to_staging`

Qué prueba:
- escritura real de Parquet para tabla principal,
- escritura opcional del auxiliar,
- y omisión limpia si el auxiliar no existe.

In [22]:
case_dir = make_case_dir("case_05_write_tables")
staging = case_dir / "staging"
staging.mkdir(parents=True, exist_ok=True)

flows_path = staging / "flows.parquet"
aux_path = staging / "flow_to_trips.parquet"

issues = []
_write_flows_table_to_staging(
    make_flows_df(),
    flows_path,
    storage_format="parquet",
    parquet_compression="snappy",
    issues=issues,
    destination_path=case_dir,
)
assert flows_path.exists()
assert issues == []

issues_aux = []
_write_optional_flow_to_trips_to_staging(
    make_flow_to_trips_df(),
    aux_path,
    write_flow_to_trips=True,
    storage_format="parquet",
    parquet_compression="snappy",
    issues=issues_aux,
    destination_path=case_dir,
)
assert aux_path.exists()
assert issues_aux == []

# Omisión limpia: no debe fallar ni escribir nada cuando el auxiliar es None.
aux_path_missing = staging / "flow_to_trips_missing.parquet"
issues_skip = []
_write_optional_flow_to_trips_to_staging(
    None,
    aux_path_missing,
    write_flow_to_trips=True,
    storage_format="parquet",
    parquet_compression="snappy",
    issues=issues_skip,
    destination_path=case_dir,
)
assert not aux_path_missing.exists()
assert issues_skip == []

show_ok("Test 4.11 - _write_flows_table_to_staging + optional aux")

OK - Test 4.11 - _write_flows_table_to_staging + optional aux


## Bloque 5. Helpers principales de Read flows

### Test 5.1 - `_validate_read_layout` happy path

Qué prueba: layout formal correcto.

In [23]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact.golondrina"
    root.mkdir(parents=True, exist_ok=True)
    paths = _resolve_flows_artifact_paths(root)
    paths.data_path.touch()
    paths.sidecar_path.touch()

    issues = []
    _validate_read_layout(root, paths, strict=False, issues=issues)
    assert issues == []

show_ok("Test 5.1 - _validate_read_layout happy path")

OK - Test 5.1 - _validate_read_layout happy path


### Test 5.2 - `_validate_read_layout` fatal por sidecar faltante

Qué prueba: `flows.metadata.json` ausente no es recuperable.

In [24]:
with tempfile.TemporaryDirectory() as td:
    root = Path(td) / "artifact.golondrina"
    root.mkdir(parents=True, exist_ok=True)

    paths = _resolve_flows_artifact_paths(root)
    paths.data_path.touch()

    issues = []
    try:
        _validate_read_layout(root, paths, strict=False, issues=issues)
        raise AssertionError("Debió fallar por sidecar faltante")
    except ExportError:
        assert_issue_present(issues, "READ_FLOWS.LAYOUT.MISSING_SIDECAR")

show_ok("Test 5.2 - _validate_read_layout missing sidecar")

OK - Test 5.2 - _validate_read_layout missing sidecar


### Test 5.3 - `_load_flow_sidecar` fatal por top-level incompleto

Qué prueba: sidecar formal exige estructura top-level mínima.

In [25]:
case_dir = make_case_dir("case_06_load_sidecar_bad")
sidecar_path = case_dir / "flows.metadata.json"

bad_payload = {
    "dataset_type": "flows",
    "format": "golondrina",
    # faltan muchas claves obligatorias
}
sidecar_path.write_text(json.dumps(bad_payload), encoding="utf-8")

issues = []
try:
    _load_flow_sidecar(
        sidecar_path,
        strict=False,
        issues=issues,
        destination_path=case_dir,
    )
    raise AssertionError("Debió fallar por top-level incompleto")
except ExportError:
    assert_issue_present(issues, "READ_FLOWS.SIDECAR.INVALID_TOP_LEVEL")

show_ok("Test 5.3 - _load_flow_sidecar invalid top-level")

OK - Test 5.3 - _load_flow_sidecar invalid top-level


### Test 5.4 - `_recover_flow_read_state` normal

Qué prueba:
- backend desde sidecar,
- preservación de ids válidos,
- snapshots cargados sin degradación.

In [26]:
payload = make_sidecar_payload(include_flow_to_trips=True)
issues = []

state = _recover_flow_read_state(
    payload,
    strict=False,
    issues=issues,
    destination_path=Path("/tmp/fake_artifact.golondrina"),
)

assert state["storage_format"] == "parquet"
assert state["dataset_id"] == "dset_sidecar"
assert state["artifact_id"] == "art_sidecar"
assert state["aggregation_spec"]["h3_resolution"] == 8
assert state["provenance"]["derived_from"][0]["type"] == "trips"
assert state["metadata"]["dataset_id"] == "dset_sidecar"
assert issues == []

show_ok("Test 5.4 - _recover_flow_read_state normal")

OK - Test 5.4 - _recover_flow_read_state normal


### Test 5.5 - expectativa correcta después de arreglar el catálogo

Qué prueba:
- regeneración de `dataset_id`,
- `artifact_id=None`,
- defaults de `aggregation_spec`, `provenance`, `metadata`,
- sin crash del render de mensajes.

In [27]:
payload = make_sidecar_payload(include_flow_to_trips=False)
payload["dataset_id"] = ""
payload["artifact_id"] = None
payload["aggregation_spec"] = None
payload["provenance"] = None
payload["metadata"] = None

issues = []
state = _recover_flow_read_state(
    payload,
    strict=False,
    issues=issues,
    destination_path=Path("/tmp/fake_artifact.golondrina"),
)

assert state["storage_format"] == "parquet"
assert isinstance(state["dataset_id"], str) and state["dataset_id"].startswith("dset_")
assert state["artifact_id"] is None
assert state["aggregation_spec"] == {}
assert state["provenance"] == {}
assert state["metadata"] == {}

assert_issue_present(issues, "READ_FLOWS.METADATA.DATASET_ID_REGENERATED")
assert_issue_present(issues, "READ_FLOWS.METADATA.ARTIFACT_ID_SET_NONE")
assert_issue_present(issues, "READ_FLOWS.SIDECAR.AGGREGATION_SPEC_DEFAULTED")
assert_issue_present(issues, "READ_FLOWS.SIDECAR.PROVENANCE_DEFAULTED")
assert_issue_present(issues, "READ_FLOWS.SIDECAR.METADATA_DEFAULTED")

show_ok("Test 5.5B - _recover_flow_read_state degraded (post-fix catálogo)")

OK - Test 5.5B - _recover_flow_read_state degraded (post-fix catálogo)



### Test 5.6 - `_read_optional_flow_to_trips` cuando no se solicita

Qué prueba: no carga y no emite issues.

In [28]:
with tempfile.TemporaryDirectory() as td:
    aux_path = Path(td) / "flow_to_trips.parquet"
    issues = []

    df_aux, loaded, files_read, n_rows = _read_optional_flow_to_trips(
        aux_path,
        requested=False,
        strict=False,
        storage_format="parquet",
        issues=issues,
        destination_path=Path(td),
    )

    assert df_aux is None
    assert loaded is False
    assert files_read == []
    assert n_rows is None
    assert issues == []

show_ok("Test 5.6 - _read_optional_flow_to_trips requested=False")

OK - Test 5.6 - _read_optional_flow_to_trips requested=False


### Test 5.7 - `_read_optional_flow_to_trips` faltante bajo `strict=False`

Qué prueba: warning + degradación a `None`.

In [29]:
with tempfile.TemporaryDirectory() as td:
    aux_path = Path(td) / "flow_to_trips.parquet"
    issues = []

    df_aux, loaded, files_read, n_rows = _read_optional_flow_to_trips(
        aux_path,
        requested=True,
        strict=False,
        storage_format="parquet",
        issues=issues,
        destination_path=Path(td),
    )

    assert df_aux is None
    assert loaded is False
    assert files_read == []
    assert n_rows is None
    assert_issue_present(issues, "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING")

show_ok("Test 5.7 - _read_optional_flow_to_trips missing strict=False")

OK - Test 5.7 - _read_optional_flow_to_trips missing strict=False


### Test 5.8 - `_read_optional_flow_to_trips` faltante bajo `strict=True`

Qué prueba: fatal cuando el auxiliar fue pedido explícitamente.


In [30]:
with tempfile.TemporaryDirectory() as td:
    aux_path = Path(td) / "flow_to_trips.parquet"
    issues = []

    try:
        _read_optional_flow_to_trips(
            aux_path,
            requested=True,
            strict=True,
            storage_format="parquet",
            issues=issues,
            destination_path=Path(td),
        )
        raise AssertionError("Debió fallar por flow_to_trips faltante con strict=True")
    except ExportError:
        assert_issue_present(issues, "READ_FLOWS.IO.FLOW_TO_TRIPS_READ_FAILED")

show_ok("Test 5.8 - _read_optional_flow_to_trips missing strict=True")

OK - Test 5.8 - _read_optional_flow_to_trips missing strict=True


### Test 5.9 - `_build_read_summary`

Qué prueba: summary mínimo estable de read.

In [31]:
summary = _build_read_summary(
    flows_df=make_flows_df(),
    flow_to_trips_loaded=True,
    n_flow_to_trips=3,
    files_read=["flows.parquet", "flow_to_trips.parquet", "flows.metadata.json"],
    dataset_id="dset_001",
    artifact_id="art_001",
)

assert summary["n_flows"] == 3
assert summary["n_columns"] == len(make_flows_df().columns)
assert summary["flow_to_trips_loaded"] is True
assert summary["n_flow_to_trips"] == 3
assert summary["files_read"] == ["flows.parquet", "flow_to_trips.parquet", "flows.metadata.json"]
assert summary["dataset_id"] == "dset_001"
assert summary["artifact_id"] == "art_001"
assert_json_dumpable(summary, "read_flows_summary")

show_ok("Test 5.9 - _build_read_summary")

OK - Test 5.9 - _build_read_summary


### Test 5.10 - `_read_flows_table` y `_read_optional_flow_to_trips` con artefactos reales

Qué prueba: lectura real de Parquet cuando el engine está disponible.

In [32]:
case_dir = make_case_dir("case_07_read_tables")
paths, payload = materialize_minimal_formal_flow_artifact(case_dir / "artifact.golondrina", with_aux=True)

issues_main = []
flows_df = _read_flows_table(
    paths.data_path,
    storage_format="parquet",
    issues=issues_main,
    destination_path=paths.root_dir,
)

assert len(flows_df) == 3
assert list(flows_df.columns) == list(make_flows_df().columns)
assert issues_main == []

issues_aux = []
flow_to_trips_df, loaded, files_read, n_rows = _read_optional_flow_to_trips(
    paths.flow_to_trips_path,
    requested=True,
    strict=False,
    storage_format="parquet",
    issues=issues_aux,
    destination_path=paths.root_dir,
)

assert loaded is True
assert n_rows == 3
assert files_read == ["flow_to_trips.parquet"]
assert list(flow_to_trips_df.columns) == list(make_flow_to_trips_df().columns)
assert issues_aux == []

show_ok("Test 5.10 - _read_flows_table + _read_optional_flow_to_trips")

OK - Test 5.10 - _read_flows_table + _read_optional_flow_to_trips


## Bloque 6: Smoke tests integrados de write_flows / read_flows
### 6.1 - Setup visible de smoke tests

Qué prepara: un directorio visible dentro de tu carpeta de trabajo para inspeccionar manualmente los artefactos persistidos de flows, y además define fixtures pequeñas para FlowDataset, sidecar y comparaciones.

In [33]:
from pathlib import Path
import json
import shutil
import copy

import pandas as pd

from pylondrina.datasets import FlowDataset
from pylondrina.errors import ExportError
from pylondrina.io.flows import (
    write_flows,
    read_flows,
    WriteFlowsOptions,
    ReadFlowsOptions,
)

SMOKE_ROOT = Path("./tmp_smoke_flows")


def reset_smoke_root() -> Path:
    if SMOKE_ROOT.exists():
        shutil.rmtree(SMOKE_ROOT)
    SMOKE_ROOT.mkdir(parents=True, exist_ok=True)
    return SMOKE_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = SMOKE_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def flow_issue_codes(report) -> list[str]:
    return [issue.code for issue in report.issues]


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def make_flow_df(n_repeat: int = 1) -> pd.DataFrame:
    base = pd.DataFrame(
        {
            "flow_id": ["f_0001", "f_0002", "f_0003"],
            "origin_h3_index": ["8828308281fffff", "8828308281fffff", "8828308285fffff"],
            "destination_h3_index": ["8828308287fffff", "8828308289fffff", "8828308287fffff"],
            "flow_count": [10, 6, 4],
            "flow_value": [10.0, 6.0, 4.0],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": pd.to_datetime(
                ["2026-01-01T08:00:00Z", "2026-01-01T08:00:00Z", "2026-01-01T09:00:00Z"],
                utc=True,
            ),
            "window_end_utc": pd.to_datetime(
                ["2026-01-01T09:00:00Z", "2026-01-01T09:00:00Z", "2026-01-01T10:00:00Z"],
                utc=True,
            ),
        }
    )
    if n_repeat <= 1:
        return base

    parts = []
    for i in range(n_repeat):
        part = base.copy(deep=True)
        part["flow_id"] = [f"{fid}_r{i:04d}" for fid in part["flow_id"]]
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def make_flow_to_trips_df(flow_ids: list[str] | None = None) -> pd.DataFrame:
    if flow_ids is None:
        flow_ids = ["f_0001", "f_0002", "f_0003"]
    rows = []
    for idx, fid in enumerate(flow_ids):
        rows.append({"flow_id": fid, "movement_id": f"m_{idx*2+1:04d}"})
        rows.append({"flow_id": fid, "movement_id": f"m_{idx*2+2:04d}"})
    return pd.DataFrame(rows)


def make_flowdataset(
    *,
    validated: bool = True,
    with_flow_to_trips: bool = False,
    dataset_id: str = "flow-dset-smoke-001",
) -> FlowDataset:
    flows_df = make_flow_df()
    flow_to_trips_df = make_flow_to_trips_df(flows_df["flow_id"].tolist()) if with_flow_to_trips else None

    aggregation_spec = {
        "h3_resolution": 8,
        "group_by": ["mode"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "min_trips_per_flow": 1,
    }
    metadata = {
        "dataset_id": dataset_id,
        "is_validated": validated,
        "events": [],
        "notes": {"smoke_case": True},
    }
    provenance = {
        "derived_from": [
            {
                "source_type": "trips",
                "dataset_id": "trip-dset-origin-001",
            }
        ],
        "prior_events_summary": {"n_events": 2},
    }

    return FlowDataset(
        flows=flows_df,
        flow_to_trips=flow_to_trips_df,
        aggregation_spec=aggregation_spec,
        source_trips={"debug_only": True},  # read_flows no debe reconstruir esto
        metadata=metadata,
        provenance=provenance,
    )


root = reset_smoke_root()
print("SMOKE_ROOT =", root.resolve())
show_ok("Test 6.1 - setup visible de smoke tests de flows")

SMOKE_ROOT = D:\Memoria\repos\pylondrina\notebooks\testing\io_flows\tmp_smoke_flows
OK - Test 6.1 - setup visible de smoke tests de flows


### Test 6.2 - smoke test de write_flows happy path mínimo

Qué prueba: escritura formal exitosa del artefacto base, sin auxiliar. Verifica bundle .golondrina, sidecar formal, summary, parameters y side effects mínimos sobre metadata.

In [34]:
case_dir = make_case_dir("case_01_write_happy")
artifact_dir = case_dir / "artifact_write_happy"
true_artifact_dir = case_dir / "artifact_write_happy.golondrina"

flows = make_flowdataset(validated=True, with_flow_to_trips=False)
flows_before = flows.flows.copy(deep=True)
metadata_before = copy.deepcopy(flows.metadata)

report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

assert report.ok is True

# Layout formal persistido
assert true_artifact_dir.exists()
assert (true_artifact_dir / "flows.parquet").exists()
assert (true_artifact_dir / "flows.metadata.json").exists()
assert not (true_artifact_dir / "flow_to_trips.parquet").exists()

# Summary y parameters mínimos
assert report.summary["n_flows"] == len(flows.flows)
assert report.summary["n_flow_to_trips"] is None
assert report.summary["dataset_id"] == flows.metadata["dataset_id"]
assert report.summary["artifact_id"] == flows.metadata["artifact_id"]
assert "flows.parquet" in report.summary["files_written"]
assert "flows.metadata.json" in report.summary["files_written"]

assert report.parameters["path"] == str(true_artifact_dir)
assert report.parameters["storage_format"] == "parquet"
assert report.parameters["mode"] == "error_if_exists"
assert report.parameters["normalize_artifact_dir"] is True
assert report.parameters["write_flow_to_trips"] is False

# Side effects mínimos en metadata
assert flows.metadata["dataset_id"] == metadata_before["dataset_id"]
assert "artifact_id" in flows.metadata
assert flows.metadata["is_validated"] is True
assert flows.metadata["events"][-1]["op"] == "write_flows"

# flows.flows no debe mutar
pd.testing.assert_frame_equal(flows.flows, flows_before)

# Sidecar consistente
sidecar = read_json(true_artifact_dir / "flows.metadata.json")
assert sidecar["dataset_type"] == "flows"
assert sidecar["format"] == "golondrina"
assert sidecar["layout_version"] == "1.1"
assert sidecar["storage"]["format"] == "parquet"
assert sidecar["files"]["data"] == "flows.parquet"
assert sidecar["files"]["metadata"] == "flows.metadata.json"
assert sidecar["files"]["flow_to_trips"] is None
assert sidecar["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["artifact_id"] == flows.metadata["artifact_id"]
assert sidecar["metadata"]["dataset_id"] == flows.metadata["dataset_id"]
assert sidecar["metadata"]["artifact_id"] == flows.metadata["artifact_id"]

display(report)
show_ok("Test 6.2 - write_flows happy path mínimo")

OperationReport(ok=True, issues=[], summary={'n_flows': 3, 'n_flow_to_trips': None, 'files_written': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_c237d25a-0269-4988-876c-6f02d4ac4020', 'path': 'tmp_smoke_flows\\case_01_write_happy\\artifact_write_happy.golondrina'}, parameters={'path': 'tmp_smoke_flows\\case_01_write_happy\\artifact_write_happy.golondrina', 'mode': 'error_if_exists', 'storage_format': 'parquet', 'parquet_compression': 'snappy', 'normalize_artifact_dir': True, 'write_flow_to_trips': False})

OK - Test 6.2 - write_flows happy path mínimo


### Test 6.3 - smoke test de categóricos persistidos eficientemente

Qué prueba: que write_flows fuerza el tratamiento categórico acotado en campos de group_by y deja evidencia observable a nivel Parquet. Además, deja una comparación visible contra una escritura manual sin dictionary encoding.

In [35]:
import pyarrow as pa
import pyarrow.parquet as pq

case_dir = make_case_dir("case_02_categorical_encoding")
artifact_dir = case_dir / "artifact_good"
artifact_bad_dir = case_dir / "artifact_bad_manual"
artifact_bad_dir.mkdir(parents=True, exist_ok=True)

flows_big = make_flowdataset(validated=True, with_flow_to_trips=False)
flows_big.flows = make_flow_df(n_repeat=3000)

report = write_flows(
    flows_big,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="overwrite",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

assert report.ok is True
good_parquet = artifact_dir / "flows.parquet"
assert good_parquet.exists()

# 1) Verificación fuerte: dictionary encoding en "mode"
parquet_file = pq.ParquetFile(good_parquet)
try:
    names = parquet_file.schema_arrow.names
    mode_idx = names.index("mode")
    encodings = {
        str(enc).upper()
        for enc in parquet_file.metadata.row_group(0).column(mode_idx).encodings
    }
    assert any("DICTIONARY" in enc for enc in encodings), encodings
finally:
    parquet_file.close()

# 2) Escritura "mala" para comparar tamaño
df_bad = flows_big.flows.copy(deep=True)
table_bad = pa.Table.from_pandas(df_bad, preserve_index=False)
pq.write_table(
    table_bad,
    artifact_bad_dir / "flows_bad_no_dictionary.parquet",
    compression="snappy",
    use_dictionary=False,
)

size_good = good_parquet.stat().st_size
size_bad = (artifact_bad_dir / "flows_bad_no_dictionary.parquet").stat().st_size

print("size_good =", size_good)
print("size_bad  =", size_bad)
print("ratio bad/good =", round(size_bad / size_good, 3) if size_good else None)

assert size_good > 0
assert size_bad > 0

show_ok("Test 6.3 - categóricos persistidos eficientemente en write_flows")

size_good = 58518
size_bad  = 74834
ratio bad/good = 1.279
OK - Test 6.3 - categóricos persistidos eficientemente en write_flows


### Test 6.4 - smoke test de read_flows happy path mínimo usando fallback .golondrina

Qué prueba: lectura formal exitosa a partir de un artefacto correcto, usando además el fallback path + ".golondrina". Verifica que no reconstruye source_trips, que fuerza is_validated=False y que deja summary/evento correctos. El contrato exige sidecar obligatorio, backend desde sidecar y source_trips=None; además, read_flows debe forzar siempre metadata["is_validated"]=False.

In [36]:
case_dir = make_case_dir("case_03_read_happy_from_sidecar")
artifact_dir = case_dir / "artifact"  # sin sufijo
true_artifact_dir = case_dir / "artifact.golondrina"

flows = make_flowdataset(validated=True, with_flow_to_trips=False)
write_report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=True,
        write_flow_to_trips=False,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,  # intencionalmente sin ".golondrina"
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=False,
    ),
)

assert write_report.ok is True
assert read_report.ok is True
assert true_artifact_dir.exists()

# Summary / parameters
assert read_report.summary["n_flows"] == len(flows.flows)
assert read_report.summary["flow_to_trips_loaded"] is False
assert read_report.summary["n_flow_to_trips"] is None
assert "flows.parquet" in read_report.summary["files_read"]
assert "flows.metadata.json" in read_report.summary["files_read"]

assert read_report.parameters["path"] == str(true_artifact_dir)
assert read_report.parameters["strict"] is False
assert read_report.parameters["keep_metadata"] is True
assert read_report.parameters["read_flow_to_trips"] is False

# Post-read
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_flows" in ops_loaded
assert ops_loaded[-1] == "read_flows"

pd.testing.assert_frame_equal(
    loaded.flows.reset_index(drop=True),
    flows.flows.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

display(read_report)
show_ok("Test 6.4 - read_flows happy path mínimo con fallback .golondrina")

OperationReport(ok=True, issues=[Issue(level='info', code='READ_FLOWS.METADATA.VALIDATED_FORCED_FALSE', message="El estado metadata['is_validated'] fue forzado a False al finalizar read_flows.", field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_flows\\case_03_read_happy_from_sidecar\\artifact.golondrina', 'strict': False, 'reason': 'force_unvalidated_after_read', 'recovered': True, 'recovery_action': 'force_is_validated_false'})], summary={'n_flows': 3, 'n_columns': 8, 'flow_to_trips_loaded': False, 'n_flow_to_trips': None, 'files_read': ['flows.parquet', 'flows.metadata.json'], 'dataset_id': 'flow-dset-smoke-001', 'artifact_id': 'art_4982f670-edaa-49ca-8fa9-d0018a105753'}, parameters={'path': 'tmp_smoke_flows\\case_03_read_happy_from_sidecar\\artifact.golondrina', 'strict': False, 'keep_metadata': True, 'read_flow_to_trips': False})

OK - Test 6.4 - read_flows happy path mínimo con fallback .golondrina


### Test 6.5 - smoke test integrado de write/read con auxiliar flow_to_trips existente

Qué prueba: persistencia y reconstrucción correcta del auxiliar opcional cuando existe y se solicita en ambas operaciones.

In [37]:
case_dir = make_case_dir("case_04_with_aux_present")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(validated=True, with_flow_to_trips=True)

write_report = write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

assert (artifact_dir / "flow_to_trips.parquet").exists()
assert "flow_to_trips.parquet" in write_report.summary["files_written"]
assert read_report.summary["flow_to_trips_loaded"] is True
assert read_report.summary["n_flow_to_trips"] == len(flows.flow_to_trips)

assert loaded.flow_to_trips is not None
pd.testing.assert_frame_equal(
    loaded.flow_to_trips.reset_index(drop=True),
    flows.flow_to_trips.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

show_ok("Test 6.5 - write/read con flow_to_trips existente")

OK - Test 6.5 - write/read con flow_to_trips existente


### Test 6.6 - smoke test degradado de read_flows con auxiliar solicitado pero faltante

Qué prueba: caso recuperable bajo strict=False. El contrato de read exige sidecar obligatorio y permite degradación controlada del auxiliar cuando se pidió pero no está en disco.

In [38]:
case_dir = make_case_dir("case_05_read_degraded_missing_aux")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(validated=True, with_flow_to_trips=True)
write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

# Dejo el artefacto formal, pero quito manualmente el auxiliar.
(artifact_dir / "flow_to_trips.parquet").unlink()

loaded, report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

codes = flow_issue_codes(report)

assert report.ok is True
assert "READ_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING" in codes
assert loaded.flow_to_trips is None
assert report.summary["flow_to_trips_loaded"] is False
assert report.summary["n_flow_to_trips"] is None

show_ok("Test 6.6 - read_flows degradado por auxiliar faltante con strict=False")

OK - Test 6.6 - read_flows degradado por auxiliar faltante con strict=False


### Test 6.7 - smoke test de read_flows con keep_metadata=False

Qué prueba: el dataset se reconstruye igual, pero no se añade el evento read_flows. El contrato vigente fija que keep_metadata=False no debe podar metadata/provenance; solo evita el append del evento.

In [39]:
case_dir = make_case_dir("case_06_read_keep_metadata_false")
artifact_dir = case_dir / "artifact"

flows = make_flowdataset(validated=True, with_flow_to_trips=False)
write_flows(
    flows,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        normalize_artifact_dir=False,
        write_flow_to_trips=False,
    ),
)

loaded, report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=False,
        read_flow_to_trips=False,
    ),
)

assert report.ok is True
assert loaded.metadata["dataset_id"] == flows.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows.metadata["artifact_id"]
assert loaded.metadata["is_validated"] is False
assert loaded.provenance == flows.provenance

ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert ops_loaded[-1] == "write_flows"  # no se agregó read_flows

show_ok("Test 6.7 - read_flows con keep_metadata=False")

OK - Test 6.7 - read_flows con keep_metadata=False


### Test 6.8 - smoke test fatal de read_flows por layout inválido

Qué prueba: artefacto con flows.parquet pero sin flows.metadata.json. La lectura formal no debe aceptarlo.

In [40]:
case_dir = make_case_dir("case_07_read_fatal_missing_sidecar")
artifact_dir = case_dir / "artifact"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Creo solo la tabla principal, sin sidecar formal.
make_flow_df().to_parquet(
    artifact_dir / "flows.parquet",
    index=False,
    compression="snappy",
    engine="pyarrow",
)

raised = None
try:
    read_flows(
        artifact_dir,
        options=ReadFlowsOptions(
            strict=False,
            keep_metadata=True,
            read_flow_to_trips=False,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ExportError)

display(raised)
show_ok("Test 6.8 - fatal de read_flows por sidecar faltante")

ExportError(message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', details={'path': 'tmp_smoke_flows\\case_07_read_fatal_missing_sidecar\\artifact', 'files_expected': ['flows.parquet', 'flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}, issue=Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, source_field=None, row_count=None, details={'path': 'tmp_smoke_flows\\case_07_read_fatal_missing_sidecar\\artifact', 'files_expected': ['flows.parquet', 'flows.metadata.json'], 'reason': 'missing_flows_metadata_json', 'action': 'abort'}), issues=(Issue(level='error', code='READ_FLOWS.LAYOUT.MISSING_SIDECAR', message='El bundle de flows no contiene flows.metadata.json; la lectura formal no es recuperable sin sidecar.', field=None, sour

OK - Test 6.8 - fatal de read_flows por sidecar faltante


### Test 6.9 - smoke test integrado de round-trip completo

Qué prueba: write + read juntos, comparando lo esencial del FlowDataset original vs el reconstruido:

- flows,
- flow_to_trips,
- aggregation_spec,
- provenance,
- dataset_id,
- artifact_id,
- eventos,
- source_trips=None,
- y la regla is_validated=False tras read.

In [42]:
case_dir = make_case_dir("case_08_roundtrip_full")
artifact_dir = case_dir / "artifact"

flows_original = make_flowdataset(validated=True, with_flow_to_trips=True)
flows_df_original = flows_original.flows.copy(deep=True)
flow_to_trips_original = flows_original.flow_to_trips.copy(deep=True)
aggregation_spec_original = copy.deepcopy(flows_original.aggregation_spec)
provenance_original = copy.deepcopy(flows_original.provenance)
print("Trips originales, flujos generados y relacion entre ellos")
display(flows_original.source_trips)
display(flows_original.flows)
display(flows_original.flow_to_trips)

write_report = write_flows(
    flows_original,
    artifact_dir,
    options=WriteFlowsOptions(
        mode="error_if_exists",
        storage_format="parquet",
        parquet_compression="snappy",
        normalize_artifact_dir=False,
        write_flow_to_trips=True,
    ),
)

loaded, read_report = read_flows(
    artifact_dir,
    options=ReadFlowsOptions(
        strict=False,
        keep_metadata=True,
        read_flow_to_trips=True,
    ),
)

assert write_report.ok is True
assert read_report.ok is True

# Identidad
assert loaded.metadata["dataset_id"] == flows_original.metadata["dataset_id"]
assert loaded.metadata["artifact_id"] == flows_original.metadata["artifact_id"]

# Regla post-read
assert loaded.metadata["is_validated"] is False
assert loaded.source_trips is None

# Data y auxiliar esencialmente iguales
pd.testing.assert_frame_equal(
    loaded.flows.reset_index(drop=True),
    flows_df_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)
pd.testing.assert_frame_equal(
    loaded.flow_to_trips.reset_index(drop=True),
    flow_to_trips_original.reset_index(drop=True),
    check_dtype=False,
    check_categorical=False,
)

# Objetos serializables equivalentes
assert loaded.aggregation_spec == aggregation_spec_original
assert loaded.provenance == provenance_original

# Eventos: write persistido + read append
ops_loaded = [ev["op"] for ev in loaded.metadata["events"]]
assert "write_flows" in ops_loaded
assert ops_loaded[-1] == "read_flows"

print("ops_loaded =", ops_loaded)
display(write_report.summary)
display(read_report.summary)

print("Datos cargados, flujos cargados y relacion entre ellos")
display(loaded.source_trips)
display(loaded.flows)
display(loaded.flow_to_trips)
show_ok("Test 6.9 - round-trip completo write_flows/read_flows")

Trips originales, flujos generados y relacion entre ellos


{'debug_only': True}

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,window_start_utc,window_end_utc
0,f_0001,8828308281fffff,8828308287fffff,10,10.0,bus,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
1,f_0002,8828308281fffff,8828308289fffff,6,6.0,metro,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
2,f_0003,8828308285fffff,8828308287fffff,4,4.0,bus,2026-01-01 09:00:00+00:00,2026-01-01 10:00:00+00:00


,flow_id,movement_id
0,f_0001,m_0001
1,f_0001,m_0002
2,f_0002,m_0003
3,f_0002,m_0004
4,f_0003,m_0005
5,f_0003,m_0006


ops_loaded = ['write_flows', 'read_flows']


...

Datos cargados, flujos cargados y relacion entre ellos


None

,flow_id,origin_h3_index,destination_h3_index,flow_count,flow_value,mode,window_start_utc,window_end_utc
0,f_0001,8828308281fffff,8828308287fffff,10,10.0,bus,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
1,f_0002,8828308281fffff,8828308289fffff,6,6.0,metro,2026-01-01 08:00:00+00:00,2026-01-01 09:00:00+00:00
2,f_0003,8828308285fffff,8828308287fffff,4,4.0,bus,2026-01-01 09:00:00+00:00,2026-01-01 10:00:00+00:00


,flow_id,movement_id
0,f_0001,m_0001
1,f_0001,m_0002
2,f_0002,m_0003
3,f_0002,m_0004
4,f_0003,m_0005
5,f_0003,m_0006


OK - Test 6.9 - round-trip completo write_flows/read_flows
